# G6M2: Baseline vs Ref-dependent (Urc1_Iref, Unified Comparator)

This notebook compares two models on the same preprocessed data:
- Baseline: `Urc1`
- Ref-dependent: `Urc1_Iref`

Goals:
- Keep Urc1_Iref core filtering logic unchanged.
- Use comparator-friendly interfaces for both models.
- Compare model performance under Low/Medium/High reference conditions with optional GT metrics.

In [10]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_Iref import Urc1_Iref
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [11]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\iref\\G6M2_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

# Shared model config
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

# Urc1_Iref-only config
IREF_EXTRA_CONFIG = {
    "leverage_threshold_factor": 5.0,
    "quantile_range": (0.01, 0.99),
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [12]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
    data_filter_h_since_last_start_min=COMMON_CONFIG["data_filter_h_since_last_start_min"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G6M2_20260501_222412.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G6M2, shape: (1096020, 3)
Time range: 2023-07-06 00:00:00 -> 2025-08-05 09:29:00

Shared preprocessed rows: 359353



In [13]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}

print("\n  -> Training Baseline (Urc1) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Baseline"] = urc_baseline
print("  OK Baseline training completed")

print("\n  -> Training Ref-dependent (Urc1_Iref) model...")
urc_iref = Urc1_Iref(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
    **IREF_EXTRA_CONFIG,
)
models["Ref-dependent (Iref)"] = urc_iref
print("  OK Urc1_Iref training completed")

print(f"\nOK {len(models)} models trained successfully\n")

coverage_rows = []
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    baseline_n = len(urc_baseline.get_Urc_results(iref))
    iref_n = len(urc_iref.get_reliable_results(iref))
    coverage_rows.append(
        {
            "Reference": ref_name,
            "Iref": iref,
            "Baseline points": baseline_n,
            "Iref points": iref_n,
        }
    )

print("Coverage overview:")
display(pd.DataFrame(coverage_rows))
urc_iref.print_summary()

STEP 2: Model Training

  -> Training Baseline (Urc1) model...
  OK Baseline training completed

  -> Training Ref-dependent (Urc1_Iref) model...

Computing interval statistics for ref-dependent filtering...
Filtering reliable intervals per Iref...
  Iref=0.28: 476 reliable intervals
  Iref=1.0: 470 reliable intervals
  Iref=1.31: 468 reliable intervals

Ref-dependent filtering complete.
  Leverage threshold factor: 5.0
  Quantile range: (0.01, 0.99)
  OK Urc1_Iref training completed

OK 2 models trained successfully

Coverage overview:


,Reference,Iref,Baseline points,Iref points
0,Low,0.28,468,476
1,Medium,1.00,468,470
2,High,1.31,468,468



Urc1_Iref Summary: Ref-Dependent Reliable Intervals
Total intervals fitted: 762
Original reliable (cond + quality): 468

Ref-dependent reliable intervals:
  Iref =  0.28 A/cm²:  476 intervals ( 62.5%)
  Iref =  1.00 A/cm²:  470 intervals ( 61.7%)
  Iref =  1.31 A/cm²:  468 intervals ( 61.4%)


In [14]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"OK UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()
    print(f"Looking for GT file: {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  OK Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  FAILED to load GT for {ref_name}: {e}")
    else:
        print(f"  MISSING GT file: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\n{'=' * 80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics enabled: {has_gt}")
print(f"{'=' * 80}\n")


STEP 3: Initialize Comparator & Load Ground Truth
OK UnifiedModelComparator initialized with 2 models

Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv
  OK Loaded GT for Low (Iref=0.28): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv
  OK Loaded GT for Medium (Iref=1.0): 758 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv
  OK Loaded GT for High (Iref=1.31): 758 points [column: gt_uref_regression]

GT status: 3/3 ref

In [15]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS (UNIFIED COMPARATOR)
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n### REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref})")

    df_metrics = comparator.compare_all(
        i_target=iref,
        outlier_threshold_method="2rmse",
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

    if df_metrics.empty:
        print("No metrics returned for this reference.")
        continue

    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print("OK Trend plot finished")
    except Exception as e:
        print(f"Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 4: Reference-Specific Metrics & Comparison

### REFERENCE CONDITION: Low (Iref=0.28, Tref=57, OHref=10)


| Model Name           |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:---------------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline             |                     0.28 |              6.216 |               468 |                   2.15303 |       3.532 |                    0 |               0.966 |              21 |          4.49 |              23.214 |          1.814 |               5.071 |           13.217 |                       539 | all_fitted   |          8.004 |         7.205 |                  468 |
| Ref-dependent (Iref) |                     0.28 |              6.379 |               476 |                   2.16835 |       3.636 |                    0 |               0.966 |              19 |          3.99 |              23.343 |          1.873 |               5.071 |           13.217 |                       539 | all_fitted   |          8.004 |         7.205 |                  468 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 0.28 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         2.153026 μV/h
  RMSE:             3.532 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.966
  Outliers:         21 (4.5%)
  Max Residual:     23.214 mV
  Mean SE:          1.814 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          8.004 mV (n=468)
  GT MAE:           7.205 mV

📊 Ref-dependent (Iref)
------------------------------------------------------------
  Data Points:      476
  Deg Rate:         2.168347 μV/h
  RMSE:             3.636 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.966
  Outliers:         19 (4.0%)
  Max Residual:     23.343 mV
  Mean SE:          1.873 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          8.004 mV (n=468)
  GT MAE:           7.205 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 Best M

| Model Name           |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:---------------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline             |                        1 |              6.216 |               468 |                   5.84556 |       6.83  |                    0 |               0.983 |              21 |          4.49 |              21.578 |          1.851 |               5.071 |           13.217 |                       539 | all_fitted   |         14.503 |        12.942 |                  468 |
| Ref-dependent (Iref) |                        1 |              6.379 |               470 |                   5.84372 |       6.825 |                    0 |               0.983 |              21 |          4.47 |              21.575 |          1.867 |               5.071 |           13.217 |                       539 | all_fitted   |         14.503 |        12.942 |                  468 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         5.845556 μV/h
  RMSE:             6.830 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.983
  Outliers:         21 (4.5%)
  Max Residual:     21.578 mV
  Mean SE:          1.851 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          14.503 mV (n=468)
  GT MAE:           12.942 mV

📊 Ref-dependent (Iref)
------------------------------------------------------------
  Data Points:      470
  Deg Rate:         5.843721 μV/h
  RMSE:             6.825 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.983
  Outliers:         21 (4.5%)
  Max Residual:     21.575 mV
  Mean SE:          1.867 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          14.503 mV (n=468)
  GT MAE:           12.942 mV

🏆 Best RMSE (vs model data):    Ref-dependent 

| Model Name           |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:---------------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline             |                     1.31 |              6.216 |               468 |                   8.46857 |      10.333 |                    0 |               0.982 |              16 |          3.42 |              44.804 |          2.249 |               5.071 |           13.217 |                       539 | all_fitted   |         13.937 |        10.982 |                  468 |
| Ref-dependent (Iref) |                     1.31 |              6.379 |               468 |                   8.46857 |      10.333 |                    0 |               0.982 |              16 |          3.42 |              44.804 |          2.249 |               5.071 |           13.217 |                       539 | all_fitted   |         13.937 |        10.982 |                  468 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.31 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         8.468567 μV/h
  RMSE:             10.333 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.982
  Outliers:         16 (3.4%)
  Max Residual:     44.804 mV
  Mean SE:          2.249 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          13.937 mV (n=468)
  GT MAE:           10.982 mV

📊 Ref-dependent (Iref)
------------------------------------------------------------
  Data Points:      468
  Deg Rate:         8.468567 μV/h
  RMSE:             10.333 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.982
  Outliers:         16 (3.4%)
  Max Residual:     44.804 mV
  Mean SE:          2.249 mV
  Cond (Median):    10^5.1
  Cond Scope:       all_fitted (n=539)
  GT RMSE:          13.937 mV (n=468)
  GT MAE:           10.982 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 

In [16]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS + SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R2 distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("OK Fit quality completed")
except Exception as e:
    print(f"Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("OK Coefficient diagnostic completed")
except Exception as e:
    print(f"Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("OK Coverage Gantt completed")
except Exception as e:
    print(f"Coverage Gantt failed: {e}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
print(f"Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"Ground truth loaded: {has_gt}")
print(f"Output plots dir: {PLOTS_OUTPUT_DIR}" if SAVE_PLOTS else "Plots displayed only")
print("\nKey settings:")
print(f"  - leverage_threshold_factor: {IREF_EXTRA_CONFIG['leverage_threshold_factor']}")
print(f"  - quantile_range: {IREF_EXTRA_CONFIG['quantile_range']}")
print(f"  - include_all_cond_metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - include_gt_metrics: {SHOW_GT_METRICS}")

if rate_tables:
    print("\nDegradation-rate summary across references:")
    display(pd.concat(rate_tables, ignore_index=True))


STEP 5: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R2 distributions)...
OK Fit quality completed

-> Plotting coefficient diagnostics...
OK Coefficient diagnostic completed

-> Plotting coverage Gantt...
OK Coverage Gantt completed

ANALYSIS COMPLETE

Models trained: 2
Reference conditions analyzed: 3
Ground truth loaded: True
Output plots dir: ..\\plots\\iref\\G6M2_comparison

Key settings:
  - leverage_threshold_factor: 5.0
  - quantile_range: (0.01, 0.99)
  - include_all_cond_metrics: True
  - include_gt_metrics: True

Degradation-rate summary across references:


,Model Name,Target Current (A/cm2),Degradation Rate (uV/h),Slope Sigma (uV/h),Reference
0,Baseline,0.28,2.153026,0.0,Low
1,Ref-dependent (Iref),0.28,2.168347,0.0,Low
2,Baseline,1.00,5.845556,0.0,Medium
3,Ref-dependent (Iref),1.00,5.843721,0.0,Medium
4,Baseline,1.31,8.468567,0.0,High
5,Ref-dependent (Iref),1.31,8.468567,0.0,High
